# Chapter 8: LiDAR SLAM — ICP, Scan Matching, Pose Graph

**SLAM Handbook Chapter 8** — LiDAR SLAMの核心であるICPをスクラッチ実装。

## このNotebookで学ぶこと

1. **ICP (Iterative Closest Point)** — Point-to-point / Point-to-plane
2. **SVDによる最適変換** — 対応点からR,tを閉形式で求める
3. **2D Scan Matching** — 2Dレーザスキャンの位置合わせ
4. **LiDAR Odometry** — 連続スキャンからの逐次推定
5. **Pose Graph SLAM** — ループクロージャによるドリフト補正

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 8.1 ICP (Iterative Closest Point)

**SLAM Handbook Section 8.2.1, Eq. 8.1**: 2つの点群P, Qの位置合わせ。

### アルゴリズム
1. P の各点に対してQ中の最近傍点を探す（対応づけ）
2. 対応点ペアから最適な $\mathbf{R}, \mathbf{t}$ を求める（SVD）
3. P を変換して繰り返す

### SVDによる最適変換 (Horn's method)
対応点 $\{(\mathbf{p}_i, \mathbf{q}_i)\}$ が与えられたとき:
$$\mathbf{H} = \sum_i (\mathbf{p}_i - \bar{\mathbf{p}})(\mathbf{q}_i - \bar{\mathbf{q}})^\top, \quad \mathbf{H} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$$
$$\mathbf{R}^* = \mathbf{V}\mathbf{U}^\top, \quad \mathbf{t}^* = \bar{\mathbf{q}} - \mathbf{R}^* \bar{\mathbf{p}}$$

In [ ]:
# =============================================================
# 2D ICP のスクラッチ実装
# =============================================================

def best_fit_transform_2d(P, Q):
    """対応点ペアから最適な R, t を SVDで求める (2D)
    P, Q: (N, 2) — 対応する点群
    """
    p_mean = P.mean(axis=0)
    q_mean = Q.mean(axis=0)
    P_centered = P - p_mean
    Q_centered = Q - q_mean
    
    H = P_centered.T @ Q_centered
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    
    # 反射を防ぐ
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    
    t = q_mean - R @ p_mean
    return R, t

def icp_2d(source, target, max_iter=50, tol=1e-6):
    """2D Point-to-Point ICP
    
    source, target: (N, 2), (M, 2) — 点群
    Returns: R, t, history
    """
    P = source.copy()
    tree = KDTree(target)
    history = []
    
    R_total = np.eye(2)
    t_total = np.zeros(2)
    
    for iteration in range(max_iter):
        # 1. 最近傍探索
        distances, indices = tree.query(P)
        Q_matched = target[indices]
        
        # 2. SVDで最適変換
        R, t = best_fit_transform_2d(P, Q_matched)
        
        # 3. 点群を変換
        P = (R @ P.T).T + t
        
        # 累積変換
        R_total = R @ R_total
        t_total = R @ t_total + t
        
        mean_error = np.mean(distances)
        history.append(mean_error)
        
        if iteration > 0 and abs(history[-2] - history[-1]) < tol:
            break
    
    return R_total, t_total, history, P

# テスト用の2D点群を生成
np.random.seed(42)

# ターゲット点群: 部屋の壁のような形状
t_wall_bottom = np.column_stack([np.linspace(0, 5, 30), np.zeros(30)])
t_wall_right = np.column_stack([np.full(20, 5), np.linspace(0, 4, 20)])
t_wall_top = np.column_stack([np.linspace(5, 2, 20), np.full(20, 4)])
t_obstacle = np.column_stack([2 + 0.5*np.cos(np.linspace(0, 2*np.pi, 15)),
                               2 + 0.5*np.sin(np.linspace(0, 2*np.pi, 15))])
target = np.vstack([t_wall_bottom, t_wall_right, t_wall_top, t_obstacle])
target += np.random.normal(0, 0.02, target.shape)

# ソース点群: ターゲットを回転+並進
theta_true = np.radians(15)
R_true = np.array([[np.cos(theta_true), -np.sin(theta_true)],
                     [np.sin(theta_true), np.cos(theta_true)]])
t_true = np.array([0.3, -0.2])
source = (R_true @ target.T).T + t_true
source += np.random.normal(0, 0.03, source.shape)

# ICP実行
R_est, t_est, history, source_aligned = icp_2d(source, target)

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(*target.T, c='blue', s=10, label='Target')
axes[0].scatter(*source.T, c='red', s=10, label='Source (misaligned)')
axes[0].set_title('Before ICP', fontweight='bold')
axes[0].legend(); axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

axes[1].scatter(*target.T, c='blue', s=10, label='Target')
axes[1].scatter(*source_aligned.T, c='green', s=10, label='Source (aligned)')
axes[1].set_title('After ICP', fontweight='bold')
axes[1].legend(); axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

axes[2].plot(history, 'b-o', markersize=3)
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Mean distance')
axes[2].set_title('ICP収束', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('2D ICP (Point-to-Point)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# 推定精度
theta_est = np.arctan2(R_est[1,0], R_est[0,0])
print(f"真の変換: θ={np.degrees(theta_true):.1f}°, t={t_true}")
print(f"推定変換: θ={np.degrees(theta_est):.1f}°, t={t_est}")
print(f"角度誤差: {np.degrees(abs(theta_true - theta_est)):.2f}°")
print(f"並進誤差: {np.linalg.norm(t_true - t_est):.4f}m")

## 8.2 LiDAR Odometry: 連続スキャンからの軌跡推定

複数のスキャンを順にICPで位置合わせし、ロボットの軌跡を推定します。

**Scan-to-scan**: 連続2フレーム間でICP → ドリフト蓄積
**Scan-to-map**: 逐次構築したローカルマップに対してICP → ドリフト軽減

In [ ]:
# =============================================================
# 2D LiDAR Odometry シミュレーション
# =============================================================
np.random.seed(123)

# 環境: 部屋の壁
walls = np.vstack([
    np.column_stack([np.linspace(0, 10, 60), np.zeros(60)]),
    np.column_stack([np.linspace(0, 10, 60), np.full(60, 8)]),
    np.column_stack([np.zeros(50), np.linspace(0, 8, 50)]),
    np.column_stack([np.full(50, 10), np.linspace(0, 8, 50)]),
    np.column_stack([np.linspace(4, 4, 20), np.linspace(0, 3, 20)]),
    np.column_stack([np.linspace(6, 8, 20), np.full(20, 5)]),
])

def simulate_2d_lidar(robot_x, robot_y, robot_th, walls, n_beams=120, max_range=8.0):
    """2D LiDARスキャンをシミュレート"""
    angles = np.linspace(-np.pi, np.pi, n_beams, endpoint=False)
    points = []
    for angle in angles:
        beam_angle = robot_th + angle
        dx, dy = np.cos(beam_angle), np.sin(beam_angle)
        min_dist = max_range
        for wx, wy in walls:
            d = np.sqrt((wx - robot_x)**2 + (wy - robot_y)**2)
            bx = robot_x + d * dx
            by = robot_y + d * dy
            if np.sqrt((bx-wx)**2 + (by-wy)**2) < 0.2:
                min_dist = min(min_dist, d)
        if min_dist < max_range:
            px = robot_x + min_dist * np.cos(beam_angle)
            py = robot_y + min_dist * np.sin(beam_angle)
            points.append([px, py])
    return np.array(points) if points else np.empty((0, 2))

# ロボットの真の軌跡
n_steps = 12
true_poses = [(2, 2, 0)]
for i in range(n_steps - 1):
    x, y, th = true_poses[-1]
    dx = 0.6 * np.cos(th) + np.random.normal(0, 0.02)
    dy = 0.6 * np.sin(th) + np.random.normal(0, 0.02)
    dth = np.random.normal(0.15, 0.05)
    true_poses.append((x+dx, y+dy, th+dth))

# 各ポーズでスキャンを取得
scans = []
for x, y, th in true_poses:
    scan = simulate_2d_lidar(x, y, th, walls)
    scan += np.random.normal(0, 0.02, scan.shape)
    scans.append(scan)

# Scan-to-scan ICP odometry
est_poses = [(2, 2, 0)]
for i in range(1, n_steps):
    R_icp, t_icp, _, _ = icp_2d(scans[i], scans[i-1], max_iter=30)
    
    x_prev, y_prev, th_prev = est_poses[-1]
    theta_icp = np.arctan2(R_icp[1,0], R_icp[0,0])
    
    # 推定ポーズの更新
    x_new = x_prev + t_icp[0]
    y_new = y_prev + t_icp[1]
    th_new = th_prev + theta_icp
    est_poses.append((x_new, y_new, th_new))

# 可視化
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.scatter(*walls.T, c='gray', s=2, alpha=0.3, label='Walls')

true_xy = np.array([(x,y) for x,y,_ in true_poses])
est_xy = np.array([(x,y) for x,y,_ in est_poses])

ax.plot(*true_xy.T, 'ko-', markersize=6, label='True trajectory')
ax.plot(*est_xy.T, 'bs--', markersize=6, label='ICP odometry')

for i, scan in enumerate(scans[::3]):
    ax.scatter(*scan.T, s=1, alpha=0.2)

ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_title('2D LiDAR Odometry (scan-to-scan ICP)', fontweight='bold')
plt.tight_layout(); plt.show()

drift = np.linalg.norm(true_xy[-1] - est_xy[-1])
total_dist = np.sum(np.linalg.norm(np.diff(true_xy, axis=0), axis=1))
print(f"総移動距離: {total_dist:.2f}m")
print(f"最終ドリフト: {drift:.4f}m ({drift/total_dist*100:.2f}%)")

## 8.3 演習問題

### 演習1: Point-to-Plane ICP
Point-to-point ICPの代わりに、ターゲット点群の法線を使ったpoint-to-plane ICPを実装してください。$d(\mathbf{p}, \mathbf{q}) = (\mathbf{n}_q \cdot (\mathbf{p} - \mathbf{q}))^2$

### 演習2: 外れ値除去
ICPの対応点ペアから距離が閾値以上のものを除去するロバストICPを実装してください。

### 演習3: 3D ICP
2D ICPを3Dに拡張してみましょう。SVDによる最適変換は同じ構造です。

## まとめ

| 要素 | 内容 |
|---|---|
| **ICP** | 最近傍対応 → SVDで最適変換 → 反復。LiDAR SLAMの基本 |
| **Distance Metric** | Point-to-point (簡単), Point-to-plane (高精度), NDT (ロバスト) |
| **Correspondence** | KD-tree, Voxel grid, Projective search |
| **LiDAR Odometry** | Scan-to-scan (ドリフト大) vs Scan-to-map (ドリフト小) |
| **Motion Distortion** | スキャン中の動きを補正。Constant-velocity / IMU / 連続時間軌道 |
| **Pose Graph** | ループクロージャでドリフトを補正（Ch01-02の手法を適用） |

### 次のNotebook
→ Ch09-12: Radar SLAM, Event Camera, IMU Preintegration, Leg Odometry